Ingest documents and chunk

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"

# Load all PDF files in the "data" directory
loader = DirectoryLoader(
    str(DATA_DIR),                
    loader_cls=PyPDFLoader       # Use PyPDFLoader for each file found
)

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
print(f"split into {len(chunks)} chunks")

split into 1230 chunks


In [3]:
import time
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_pinecone import PineconeVectorStore  # or your respective import
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# (Assuming 'documents' are already loaded above this point)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    output_dimensionality=768,  # Matches your Pinecone index dimension
)

index_name = "av-paper-index"

# 1. Initialize the vector store connection without uploading everything at once
vector_store = PineconeVectorStore(
    index_name=index_name, embedding=embeddings
)

# 2. Upload chunks in controlled batches with a pause to prevent 429 errors
batch_size = 10  # Adjust between 5 to 20 depending on your quota limits
total_batches = (len(chunks) + batch_size - 1) // batch_size

for i in range(0, len(chunks), batch_size):
  batch = chunks[i : i + batch_size]
  vector_store.add_documents(batch)
  print(
      f"Successfully uploaded batch {i // batch_size + 1} of {total_batches}"
  )
  time.sleep(2)  # Pause for 2 seconds between batches

# 3. Run your similarity search query
query = "What are the main findings in the paper?"
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
  print(f"--- Result {i+1} (Page {doc.metadata.get('page', 'N/A')}) ---")
  print(doc.page_content)
  print("\n")

Successfully uploaded batch 1 of 80
Successfully uploaded batch 2 of 80
Successfully uploaded batch 3 of 80
Successfully uploaded batch 4 of 80


KeyboardInterrupt: 